# SmartOrder ML - Phase 4 Colab Training

Notebook autonome pour entrainer les prototypes de classification et, si disponible, de regression sur `data/sandbox/train.parquet` et `data/sandbox/test.parquet`.

**Point critique :** le dataset contient environ 97% de donnees synthetiques. La classe `late_blocking` derive d'un seul exemple reel duplique avec bruit. Toutes les metriques ci-dessous doivent donc etre lues comme des signaux exploratoires, pas comme une validation production.

## 1. Setup

Installation explicite des dependances d'entrainement. `pyarrow` est ajoute explicitement parce que les splits sont au format Parquet.

In [ ]:
CORE_DEPENDENCIES = [
    'pandas==2.2.3',
    'numpy==1.26.4',
    'scikit-learn==1.5.2',
    'xgboost==2.1.3',
    'catboost==1.2.7',
    'imbalanced-learn==0.12.4',
    'sdv==1.17.2',
    'shap==0.46.0',
    'mlflow==2.18.0',
    'fastapi==0.115.6',
    'uvicorn[standard]==0.32.1',
    'pydantic==2.10.3',
    'pydantic-settings==2.7.0',
    'hdbcli==2.22.32',
    'sqlalchemy==2.0.36',
    'psycopg2-binary==2.9.10',
    'httpx==0.28.1',
    'python-dotenv==1.0.1',
    'pyyaml==6.0.2',
    'joblib==1.4.2',
    'pyarrow>=16.0.0',  # Ecart documente: necessaire pour lire train/test.parquet.
]

%pip install -q {' '.join(CORE_DEPENDENCIES)}

Clone ou reutilisation du repo, puis installation editable de `ml-service` pour importer `features.feature_engineer` sans recopier le code source dans le notebook.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('SMARTORDER_REPO_URL', 'https://github.com/SalahEddineAbaid/SAPBTP.git')
REPO_DIR = Path('/content/SAPBTP')
ML_SERVICE_DIR = REPO_DIR / 'ml-service'

if not ML_SERVICE_DIR.exists():
    if REPO_DIR.exists():
        print(f'Repo directory exists but {ML_SERVICE_DIR} is missing: {REPO_DIR}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

if not ML_SERVICE_DIR.exists():
    raise FileNotFoundError(f'Cannot find ml-service at {ML_SERVICE_DIR}. Set SMARTORDER_REPO_URL if the repo moved.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ML_SERVICE_DIR)], check=True)
sys.path.insert(0, str(ML_SERVICE_DIR))
print(f'Using ml-service from: {ML_SERVICE_DIR}')

Chargement des splits et des configurations existantes du repo.

In [ ]:
import json
import warnings
from datetime import date

import mlflow
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display

from features.feature_engineer import FeatureEngineer

DATA_DIR = ML_SERVICE_DIR / 'data' / 'sandbox'
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
MODEL_CONFIG_PATH = ML_SERVICE_DIR / 'config' / 'model_config.yaml'
FEATURE_CONFIG_PATH = ML_SERVICE_DIR / 'config' / 'feature_config.yaml'

for required_path in [TRAIN_PATH, TEST_PATH, MODEL_CONFIG_PATH, FEATURE_CONFIG_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)
model_config = yaml.safe_load(MODEL_CONFIG_PATH.read_text(encoding='utf-8'))
feature_config = yaml.safe_load(FEATURE_CONFIG_PATH.read_text(encoding='utf-8'))

print(f'Train shape: {train_raw.shape}')
print(f'Test shape: {test_raw.shape}')
print(f'Model config: {MODEL_CONFIG_PATH}')
print(f'Feature config: {FEATURE_CONFIG_PATH}')
display(train_raw.head())

## 2. Verification pre-entrainement

Ces controles sont executes avant tout `fit`: distribution des classes, croisement source/classe, puis verification de lignes identiques ou quasi-identiques entre train et test.

In [ ]:
TARGET_CLASS = model_config['classification']['target']
TARGET_DELAY = model_config.get('regression', {}).get('target', 'target_delay_days')

for df_name, df in [('train', train_raw), ('test', test_raw)]:
    missing = {TARGET_CLASS, 'data_source'} - set(df.columns)
    if missing:
        raise ValueError(f'{df_name} is missing required columns: {sorted(missing)}')

print('Distribution target_class - train')
display(train_raw[TARGET_CLASS].value_counts(dropna=False).rename_axis(TARGET_CLASS).to_frame('count'))

print('Distribution target_class - test')
display(test_raw[TARGET_CLASS].value_counts(dropna=False).rename_axis(TARGET_CLASS).to_frame('count'))

print('Distribution croisee data_source x target_class - train')
display(pd.crosstab(train_raw['data_source'], train_raw[TARGET_CLASS], dropna=False))

print('Distribution croisee data_source x target_class - test')
display(pd.crosstab(test_raw['data_source'], test_raw[TARGET_CLASS], dropna=False))

real_data_ratio = float((pd.concat([train_raw, test_raw], ignore_index=True)['data_source'] == 'real').mean())
print(f'Real data ratio total: {real_data_ratio:.4f}')

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

def assert_no_train_test_overlap(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    excluded = {TARGET_CLASS, TARGET_DELAY, 'data_source'}
    common_cols = [c for c in train_df.columns if c in test_df.columns and c not in excluded]
    if not common_cols:
        raise ValueError('No common columns available for train/test overlap verification.')

    train_exact = train_df[common_cols].astype(str).fillna('<NA>').agg('|'.join, axis=1)
    test_exact = test_df[common_cols].astype(str).fillna('<NA>').agg('|'.join, axis=1)
    exact_test_mask = test_exact.isin(set(train_exact))
    if exact_test_mask.any():
        display(test_df.loc[exact_test_mask, common_cols + [TARGET_CLASS, 'data_source']].head(10))
        raise AssertionError(f'{int(exact_test_mask.sum())} test rows are exactly identical to train rows.')

    fe_check = FeatureEngineer(reference_date=date.today())
    train_feat = fe_check.transform_dataframe(train_df)[fe_check.get_feature_names()]
    test_feat = fe_check.transform_dataframe(test_df)[fe_check.get_feature_names()]
    train_enc = fe_check.encode_categoricals_for_sklearn(train_feat)
    test_enc = fe_check.encode_categoricals_for_sklearn(test_feat)
    train_enc, test_enc = train_enc.align(test_enc, join='outer', axis=1, fill_value=0)
    train_enc = train_enc.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    test_enc = test_enc.apply(pd.to_numeric, errors='coerce').fillna(0.0)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_enc)
    test_scaled = scaler.transform(test_enc)
    distances = pairwise_distances(test_scaled, train_scaled, metric='euclidean')
    nearest = distances.min(axis=1)
    quasi_threshold = 1e-6
    quasi_mask = nearest <= quasi_threshold
    if quasi_mask.any():
        suspect = test_df.loc[quasi_mask, common_cols + [TARGET_CLASS, 'data_source']].copy()
        suspect['nearest_train_distance'] = nearest[quasi_mask]
        display(suspect.head(10))
        raise AssertionError(f'{int(quasi_mask.sum())} test rows are quasi-identical to train rows after feature engineering.')

    print('Overlap check passed: no exact or quasi-identical test rows found in train.')
    print(f'Closest normalized engineered-feature distance: {nearest.min():.8f}')

assert_no_train_test_overlap(train_raw, test_raw)

## 3. Entrainement

Entrainement de trois modeles de classification: Logistic Regression, XGBoost et CatBoost. Les hyperparametres viennent de `model_config.yaml`; les seules adaptations documentees sont le nombre de folds, borne par le volume disponible par classe, et la traduction de `cat_features: auto` en colonnes categoricielles reelles.

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'file:///content/mlruns'))
mlflow.set_experiment(os.environ.get('MLFLOW_EXPERIMENT_NAME', 'smartorder-ml-colab'))

fe = FeatureEngineer(reference_date=date.today())
feature_names = fe.get_feature_names()
numeric_features = fe.get_numeric_feature_names()
categorical_features = fe.get_categorical_feature_names()
catboost_cat_features = feature_config.get('catboost_cat_features', fe.CATBOOST_CAT_COLS)

train_features = fe.transform_dataframe(train_raw)
test_features = fe.transform_dataframe(test_raw)
X_train_raw = train_features[feature_names].copy()
X_test_raw = test_features[feature_names].copy()
y_train = train_features[TARGET_CLASS].astype(str)
y_test = test_features[TARGET_CLASS].astype(str)

for col in categorical_features:
    X_train_raw[col] = X_train_raw[col].fillna('').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('').astype(str)

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)
class_labels = list(label_encoder.classes_)

min_class_count = int(y_train.value_counts().min())
configured_splits = int(model_config['classification']['cv_strategy'].get('n_splits', 5))
n_splits = min(5, configured_splits, min_class_count)
if n_splits < 3:
    raise ValueError(f'Need at least 3 samples per class for stratified CV; minimum class count is {min_class_count}.')
n_repeats = int(model_config['classification']['cv_strategy'].get('n_repeats', 1))
random_state = int(model_config['classification']['cv_strategy'].get('random_state', 42))
cv_clf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
print(f'Classification CV: RepeatedStratifiedKFold(n_splits={n_splits}, n_repeats={n_repeats})')

preprocess_linear = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)
preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

def clean_params(params: dict, forbidden: set[str] | None = None) -> dict:
    forbidden = forbidden or set()
    return {k: v for k, v in params.items() if k not in forbidden and v != 'auto'}

clf_cfg = model_config['classification']['models']
logreg_params = clean_params(clf_cfg['logistic_regression']['params'])
xgb_params = clean_params(clf_cfg['xgboost']['params'], forbidden={'use_label_encoder'})
cat_params = clean_params(clf_cfg['catboost']['params'], forbidden={'cat_features'})

classifiers = {
    'logistic_regression': Pipeline([('preprocess', preprocess_linear), ('model', LogisticRegression(**logreg_params))]),
    'xgboost': Pipeline([('preprocess', preprocess_tree), ('model', XGBClassifier(**xgb_params))]),
    'catboost': CatBoostClassifier(**cat_params, cat_features=catboost_cat_features),
}

classification_runs = []
for model_name, estimator in classifiers.items():
    X_for_model = X_train_raw if model_name == 'catboost' else X_train_raw
    with mlflow.start_run(run_name=f'clf-{model_name}') as run:
        mlflow.set_tags({
            'task': 'classification',
            'model_name': model_name,
            'real_data_ratio': f'{real_data_ratio:.6f}',
            'prototype_status': 'exploratory',
            'dataset_warning': '97_percent_synthetic_late_blocking_from_one_real_example',
        })
        mlflow.log_param('cv_n_splits_effective', n_splits)
        mlflow.log_param('cv_n_repeats', n_repeats)
        mlflow.log_param('feature_schema_version', '1.0')
        mlflow.log_params({f'param_{k}': v for k, v in (clf_cfg[model_name]['params']).items()})
        if model_name == 'xgboost':
            mlflow.log_param('documented_deviation_use_label_encoder', 'removed because xgboost>=2 handles labels without this deprecated parameter')
        if model_name == 'catboost':
            mlflow.log_param('documented_deviation_cat_features', ','.join(catboost_cat_features))

        cv_scores = cross_val_score(clone(estimator), X_for_model, y_train_enc, cv=cv_clf, scoring='f1_macro', n_jobs=None)
        fitted = clone(estimator)
        fitted.fit(X_for_model, y_train_enc)
        y_pred_enc = fitted.predict(X_test_raw)
        y_pred_enc = np.asarray(y_pred_enc).reshape(-1).astype(int)
        test_f1 = f1_score(y_test_enc, y_pred_enc, average='macro', zero_division=0)

        mlflow.log_metric('cv_f1_macro_mean', float(cv_scores.mean()))
        mlflow.log_metric('cv_f1_macro_std', float(cv_scores.std()))
        mlflow.log_metric('test_f1_macro', float(test_f1))
        mlflow.sklearn.log_model(fitted, artifact_path='model')

        classification_runs.append({
            'model_name': model_name,
            'run_id': run.info.run_id,
            'model_uri': f'runs:/{run.info.run_id}/model',
            'estimator': fitted,
            'cv_f1_macro_mean': float(cv_scores.mean()),
            'cv_f1_macro_std': float(cv_scores.std()),
            'test_f1_macro': float(test_f1),
            'y_pred_enc': y_pred_enc,
        })
        print(f'{model_name}: CV F1-macro={cv_scores.mean():.4f} +/- {cv_scores.std():.4f}; test F1-macro={test_f1:.4f}')

classification_summary = pd.DataFrame([{k: v for k, v in r.items() if k not in {'estimator', 'y_pred_enc'}} for r in classification_runs])
display(classification_summary.sort_values('test_f1_macro', ascending=False))

## 4. Evaluation en deux temps

Evaluation globale sur tout `test.parquet`, puis evaluation filtree sur `data_source == 'real'` avec avertissement si le volume est trop faible.

In [ ]:
best_clf = max(classification_runs, key=lambda r: r['test_f1_macro'])
best_y_pred_enc = best_clf['y_pred_enc']
best_y_pred = label_encoder.inverse_transform(best_y_pred_enc)

global_f1_macro = f1_score(y_test, best_y_pred, average='macro', labels=class_labels, zero_division=0)
print(f"Selected classifier by F1-macro: {best_clf['model_name']}")
print(f'Global test F1-macro: {global_f1_macro:.4f}')
print('\nClassification report - full test')
print(classification_report(y_test, best_y_pred, labels=class_labels, zero_division=0))

cm = confusion_matrix(y_test, best_y_pred, labels=class_labels)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot(xticks_rotation=45)

real_mask = test_raw['data_source'].eq('real')
real_test_rows = int(real_mask.sum())
if real_test_rows == 0:
    real_f1_macro = np.nan
    print('WARNING: no real rows are present in test.parquet; real-only F1 is undefined.')
else:
    real_f1_macro = f1_score(y_test[real_mask], best_y_pred[real_mask], average='macro', labels=class_labels, zero_division=0)
    print(f'Real-only test rows: {real_test_rows}')
    print(f'Real-only test F1-macro: {real_f1_macro:.4f}')
    print('\nClassification report - real-only test')
    print(classification_report(y_test[real_mask], best_y_pred[real_mask], labels=class_labels, zero_division=0))
    if real_test_rows < 10:
        print('WARNING: the real-only test sample is too small to be statistically meaningful.')

report_dict = classification_report(y_test, best_y_pred, labels=class_labels, output_dict=True, zero_division=0)
class_sources = pd.crosstab(pd.concat([train_raw, test_raw])['target_class'], pd.concat([train_raw, test_raw])['data_source'])
if 'late_non_blocking' in report_dict:
    late_non_blocking_real = int(class_sources.get('real', pd.Series(dtype=int)).get('late_non_blocking', 0))
    late_non_blocking_f1 = float(report_dict['late_non_blocking']['f1-score'])
    if late_non_blocking_real == 0 and late_non_blocking_f1 > 0.95:
        print('ALERT: late_non_blocking exceeds 95% F1 while this class is 100% synthetic. Interpret as overfit to generation rules, not real performance.')

## 5. Regression si `target_delay_days` est disponible

Modele de regression separe avec MAE/RMSE globales et sur donnees reelles uniquement.

In [ ]:
regression_runs = []
best_reg = None
global_reg_metrics = None
real_reg_metrics = None

if TARGET_DELAY not in train_features.columns or train_features[TARGET_DELAY].isna().all():
    print(f'Regression skipped: {TARGET_DELAY} is absent or empty.')
else:
    reg_train_mask = train_features[TARGET_DELAY].notna()
    reg_test_mask = test_features[TARGET_DELAY].notna()
    X_train_reg = X_train_raw.loc[reg_train_mask]
    y_train_reg = pd.to_numeric(train_features.loc[reg_train_mask, TARGET_DELAY], errors='coerce')
    X_test_reg = X_test_raw.loc[reg_test_mask]
    y_test_reg = pd.to_numeric(test_features.loc[reg_test_mask, TARGET_DELAY], errors='coerce')

    reg_cfg = model_config['regression']['models']
    reg_splits = min(5, int(model_config['regression']['cv_strategy'].get('n_splits', 5)), len(X_train_reg))
    if reg_splits < 3:
        raise ValueError(f'Need at least 3 rows for regression CV; found {len(X_train_reg)}.')
    reg_repeats = int(model_config['regression']['cv_strategy'].get('n_repeats', 1))
    cv_reg = RepeatedKFold(n_splits=reg_splits, n_repeats=reg_repeats, random_state=random_state)

    reg_models = {
        'elasticnet': Pipeline([('preprocess', preprocess_linear), ('model', ElasticNet(**clean_params(reg_cfg['elasticnet']['params'])))]),
        'xgboost_regressor': Pipeline([('preprocess', preprocess_tree), ('model', XGBRegressor(**clean_params(reg_cfg['xgboost_regressor']['params'])))]),
        'catboost_regressor': CatBoostRegressor(**clean_params(reg_cfg['catboost_regressor']['params'], forbidden={'cat_features'}), cat_features=catboost_cat_features),
    }

    for model_name, estimator in reg_models.items():
        X_for_model = X_train_reg
        with mlflow.start_run(run_name=f'reg-{model_name}') as run:
            mlflow.set_tags({
                'task': 'regression',
                'model_name': model_name,
                'real_data_ratio': f'{real_data_ratio:.6f}',
                'prototype_status': 'exploratory',
            })
            mlflow.log_params({f'param_{k}': v for k, v in reg_cfg[model_name]['params'].items()})
            if model_name == 'catboost_regressor':
                mlflow.log_param('documented_deviation_cat_features', ','.join(catboost_cat_features))

            cv_mae = -cross_val_score(clone(estimator), X_for_model, y_train_reg, cv=cv_reg, scoring='neg_mean_absolute_error')
            fitted = clone(estimator)
            fitted.fit(X_for_model, y_train_reg)
            pred = fitted.predict(X_test_reg)
            mae = mean_absolute_error(y_test_reg, pred)
            rmse = mean_squared_error(y_test_reg, pred, squared=False)
            mlflow.log_metric('cv_mae_mean', float(cv_mae.mean()))
            mlflow.log_metric('cv_mae_std', float(cv_mae.std()))
            mlflow.log_metric('test_mae', float(mae))
            mlflow.log_metric('test_rmse', float(rmse))
            mlflow.sklearn.log_model(fitted, artifact_path='model')
            regression_runs.append({
                'model_name': model_name,
                'run_id': run.info.run_id,
                'model_uri': f'runs:/{run.info.run_id}/model',
                'estimator': fitted,
                'test_mae': float(mae),
                'test_rmse': float(rmse),
                'cv_mae_mean': float(cv_mae.mean()),
                'pred': pred,
            })
            print(f'{model_name}: CV MAE={cv_mae.mean():.4f}; test MAE={mae:.4f}; RMSE={rmse:.4f}')

    best_reg = min(regression_runs, key=lambda r: r['test_mae'])
    global_reg_metrics = {'mae': best_reg['test_mae'], 'rmse': best_reg['test_rmse']}
    real_reg_mask = real_mask.loc[reg_test_mask].to_numpy()
    if real_reg_mask.sum() == 0:
        real_reg_metrics = {'mae': np.nan, 'rmse': np.nan, 'rows': 0}
        print('WARNING: no real rows available for regression evaluation.')
    else:
        real_mae = mean_absolute_error(y_test_reg.to_numpy()[real_reg_mask], best_reg['pred'][real_reg_mask])
        real_rmse = mean_squared_error(y_test_reg.to_numpy()[real_reg_mask], best_reg['pred'][real_reg_mask], squared=False)
        real_reg_metrics = {'mae': float(real_mae), 'rmse': float(real_rmse), 'rows': int(real_reg_mask.sum())}
        print(f"Best regression model: {best_reg['model_name']}")
        print(f'Regression real-only rows: {real_reg_metrics["rows"]}; MAE={real_mae:.4f}; RMSE={real_rmse:.4f}')

    display(pd.DataFrame([{k: v for k, v in r.items() if k not in {'estimator', 'pred'}} for r in regression_runs]).sort_values('test_mae'))

## 6. Selection et export MLflow Registry

Selection du classifieur par F1-macro, puis enregistrement dans MLflow Registry avec les tags requis par `training/README.md` et les tags de statut prototype.

In [ ]:
client = mlflow.tracking.MlflowClient()

registered_clf_name = os.environ.get('SMARTORDER_CLF_MODEL_NAME', 'smartorder-clf')
clf_registration = mlflow.register_model(best_clf['model_uri'], registered_clf_name)
clf_final_uri = f'models:/{registered_clf_name}/{clf_registration.version}'

clf_tags = {
    'feature_schema_version': '1.0',
    'training_rows': str(len(X_train_raw)),
    'cv_f1_macro': str(round(best_clf['cv_f1_macro_mean'], 4)),
    'test_f1_macro': str(round(global_f1_macro, 4)),
    'real_test_f1_macro': 'nan' if pd.isna(real_f1_macro) else str(round(float(real_f1_macro), 4)),
    'real_test_rows': str(real_test_rows),
    'real_data_ratio': f'{real_data_ratio:.6f}',
    'prototype_status': 'exploratory',
    'selected_by': 'test_f1_macro_not_accuracy',
    'dataset_warning': '97_percent_synthetic_late_blocking_from_one_real_example',
}
for key, value in clf_tags.items():
    client.set_model_version_tag(registered_clf_name, clf_registration.version, key, value)

print(f'Registered classifier URI: {clf_final_uri}')

reg_final_uri = None
if best_reg is not None:
    registered_reg_name = os.environ.get('SMARTORDER_REG_MODEL_NAME', 'smartorder-reg')
    reg_registration = mlflow.register_model(best_reg['model_uri'], registered_reg_name)
    reg_final_uri = f'models:/{registered_reg_name}/{reg_registration.version}'
    reg_tags = {
        'feature_schema_version': '1.0',
        'training_rows': str(len(X_train_reg)),
        'cv_mae': str(round(best_reg['cv_mae_mean'], 4)),
        'test_mae': str(round(best_reg['test_mae'], 4)),
        'test_rmse': str(round(best_reg['test_rmse'], 4)),
        'real_data_ratio': f'{real_data_ratio:.6f}',
        'prototype_status': 'exploratory',
    }
    for key, value in reg_tags.items():
        client.set_model_version_tag(registered_reg_name, reg_registration.version, key, value)
    print(f'Registered regressor URI: {reg_final_uri}')

## 7. Rapport de fin de notebook

La cellule suivante genere le rapport final en Markdown avec les valeurs calculees pendant l'execution.

In [ ]:
regression_line = 'Regression non executee: `target_delay_days` absent ou vide.'
if best_reg is not None:
    regression_line = (
        f"Regresseur selectionne: `{best_reg['model_name']}` par MAE minimale. "
        f"Global MAE={global_reg_metrics['mae']:.4f}, RMSE={global_reg_metrics['rmse']:.4f}; "
        f"reel uniquement ({real_reg_metrics['rows']} lignes): MAE={real_reg_metrics['mae']:.4f}, RMSE={real_reg_metrics['rmse']:.4f}. "
        f"URI Registry: `{reg_final_uri}`."
    )

final_report = f'''
### Rapport final SmartOrder Phase 4

- **Classifieur selectionne:** `{best_clf['model_name']}`, choisi par F1-macro sur test (`{global_f1_macro:.4f}`), pas par accuracy globale.
- **Performance globale:** F1-macro test = `{global_f1_macro:.4f}`.
- **Performance sur donnees reelles uniquement:** F1-macro = `{'nan' if pd.isna(real_f1_macro) else f'{real_f1_macro:.4f}'}` sur `{real_test_rows}` exemples reels en test.
- **URI MLflow Registry classifieur:** `{clf_final_uri}`.
- **Regression:** {regression_line}
- **Limite principale:** environ 97% du dataset est synthetique, et `late_blocking` derive d'un seul exemple reel duplique avec bruit.
- **Recommandation:** ne pas deployer ce modele en decision automatique tant que le volume de donnees reelles annotees n'augmente pas substantiellement.
'''

display(Markdown(final_report))
print('Answers required by validation:')
print(f'1. Global F1-macro={global_f1_macro:.4f}; real-only F1-macro={real_f1_macro}')
print(f'2. Classifier Registry URI={clf_final_uri}')
print(f'3. Real examples in test={real_test_rows}')